In [1]:
import pandas as pd

RUN_DATA = "paper_results/Master_RunLevel_Data_NewJudge.csv"
AGGREDATED_DATA = "paper_results/Master_Aggregated_Data_NewJudge.csv"

print("Loading run-level CSV...")
df = pd.read_csv(RUN_DATA)
df

df = pd.read_csv(RUN_DATA)
    # 4. Recalculate the Final EE Tables
target_cols = ['fidelity_passed']

print("--- 1. NEW FILTERED EE (By Model & Prompt) ---")
agg_model_prompt = df.groupby(['model', 'prompt'])[target_cols].mean() * 100
print(agg_model_prompt.round(2).to_string())

print("--- 1. NEW FILTERED EE (By Model & Prompt) ---")
# agg_model_prompt = df.groupby(['model', 'prompt', 'vignette_id'])[target_cols].mean() * 100
# print(agg_model_prompt.round(2).to_string())
# agg_model_prompt.to_csv("EE_Model_Prompt_Filtered_new.csv")


import json

def n_contradicted(p):
    try:
        probs = json.loads(p) if isinstance(p, str) else (p or [])
        return sum(1 for x in probs if x.get("type") == "CONTRADICTED")
    except Exception:
        return float("nan")

df["has_contradiction"] = (df["fidelity_problems"].apply(n_contradicted) > 0).astype(float)
df.loc[df["fidelity_passed"].isna(), "has_contradiction"] = float("nan")
print((df.groupby(["model"])["has_contradiction"].mean() * 100).round(1))


Loading run-level CSV...


/tmp/ipykernel_3553546/3493108520.py:7: DtypeWarning: Columns (0: race_quote) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(RUN_DATA)
/tmp/ipykernel_3553546/3493108520.py:10: DtypeWarning: Columns (0: race_quote) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(RUN_DATA)


--- 1. NEW FILTERED EE (By Model & Prompt) ---
                      fidelity_passed
model     prompt                     
llama     acknowledge       61.207041
          baseline          61.250314
          ignore            61.616679
med_gemma acknowledge       31.720834
          baseline          31.748711
          ignore            31.941467
qwen      acknowledge       68.968127
          baseline          70.078408
          ignore            70.016003
--- 1. NEW FILTERED EE (By Model & Prompt) ---
model
llama        35.7
med_gemma    57.2
qwen         23.9
Name: has_contradiction, dtype: float64


In [4]:
import json
import numpy as np
import pandas as pd

df = pd.read_csv("paper_results/gemma/Master_RunLevel_Data_gemma_NewJudge.csv", low_memory=False)

# --- parse the judge's problem list ---
def parse(p):
    if isinstance(p, str):
        try:
            return json.loads(p)
        except Exception:
            return None
    return None

df["problems"] = df["fidelity_problems"].apply(parse)
judged = df["problems"].notna()          # runs the judge could evaluate

def has_type(probs, t):
    return float(any(x.get("type") == t for x in probs)) if isinstance(probs, list) else np.nan

df["any_problem"]  = df["problems"].apply(lambda p: float(len(p) > 0) if isinstance(p, list) else np.nan)
df["contradicted"] = df["problems"].apply(lambda p: has_type(p, "CONTRADICTED"))
df["fabricated"]   = df["problems"].apply(lambda p: has_type(p, "FABRICATED"))
df.loc[~judged, ["any_problem", "contradicted", "fabricated"]] = np.nan

cols = ["any_problem", "contradicted", "fabricated"]
print("=== % of explanations with at least one problem, by model & prompt ===")
print((df.groupby(["model", "prompt"])[cols].mean() * 100).round(1))

print("\n=== by model (pooled over prompts) ===")
print((df.groupby("model")[cols].mean() * 100).round(1))

print("\n=== contradicted-only, by model & vignette (pooled over prompts) ===")
print((df.groupby(["model", "vignette_id"])["contradicted"].mean() * 100).round(1).unstack(0))

# --- export flagged examples for the two manual checks ---
check = df[judged & (df["any_problem"] == 1) &
           (((df.model == "qwen") & (df.vignette_id == 1)) | (df.vignette_id == 5))]
sample = check.groupby(["model", "vignette_id"]).head(10)
rows = []
for r in sample.itertuples():
    for p in r.problems:
        rows.append({"model": r.model, "vignette_id": r.vignette_id, "prompt": r.prompt,
                     "photo_id": r.photo_id, "response_n": r.response_n,
                     "type": p.get("type"), "quote": p.get("quote"),
                     "case_text_quote": p.get("case_text_quote")})
# pd.DataFrame(rows).to_csv("fidelity_flags_to_check.csv", index=False)
print(f"\nExported {len(rows)} flagged statements to fidelity_flags_to_check.csv")

=== % of explanations with at least one problem, by model & prompt ===
                   any_problem  contradicted  fabricated
model prompt                                            
gemma acknowledge         73.4          68.4        24.3
      baseline            72.8          67.6        23.3
      ignore              70.5          66.5        21.6

=== by model (pooled over prompts) ===
       any_problem  contradicted  fabricated
model                                       
gemma         72.3          67.5        23.1

=== contradicted-only, by model & vignette (pooled over prompts) ===
model        gemma
vignette_id       
1             50.5
2             72.3
3             77.7
4             26.6
5             90.0
6             98.4
7             99.9
8             99.7
9             99.6
10            39.3
11            24.3
12            33.7

Exported 27 flagged statements to fidelity_flags_to_check.csv


In [6]:
import json
import numpy as np
import pandas as pd

df = pd.read_csv("paper_results/Master_RunLevel_Data_NewJudge.csv", low_memory=False)

# --- parse the judge's problem list ---
def parse(p):
    if isinstance(p, str):
        try:
            return json.loads(p)
        except Exception:
            return None
    return None

df["problems"] = df["fidelity_problems"].apply(parse)
judged = df["problems"].notna()          # runs the judge could evaluate

def has_type(probs, t):
    return float(any(x.get("type") == t for x in probs)) if isinstance(probs, list) else np.nan

df["any_problem"]  = df["problems"].apply(lambda p: float(len(p) > 0) if isinstance(p, list) else np.nan)
df["contradicted"] = df["problems"].apply(lambda p: has_type(p, "CONTRADICTED"))
df["fabricated"]   = df["problems"].apply(lambda p: has_type(p, "FABRICATED"))
df.loc[~judged, ["any_problem", "contradicted", "fabricated"]] = np.nan

cols = ["any_problem", "contradicted", "fabricated"]
print("=== % of explanations with at least one problem, by model & prompt ===")
print((df.groupby(["model", "prompt"])[cols].mean() * 100).round(1))

print("\n=== by model (pooled over prompts) ===")
print((df.groupby("model")[cols].mean() * 100).round(1))

print("\n=== contradicted-only, by model & vignette (pooled over prompts) ===")
print((df.groupby(["model", "vignette_id"])["contradicted"].mean() * 100).round(1).unstack(0))

# --- export flagged examples for the two manual checks ---
check = df[judged & (df["any_problem"] == 1) &
           (((df.model == "qwen") & (df.vignette_id == 1)) | (df.vignette_id == 5))]
sample = check.groupby(["model", "vignette_id"]).head(10)
rows = []
for r in sample.itertuples():
    for p in r.problems:
        rows.append({"model": r.model, "vignette_id": r.vignette_id, "prompt": r.prompt,
                     "photo_id": r.photo_id, "response_n": r.response_n,
                     "type": p.get("type"), "quote": p.get("quote"),
                     "case_text_quote": p.get("case_text_quote")})
# pd.DataFrame(rows).to_csv("fidelity_flags_to_check.csv", index=False)
print(f"\nExported {len(rows)} flagged statements to fidelity_flags_to_check.csv")

=== % of explanations with at least one problem, by model & prompt ===
                       any_problem  contradicted  fabricated
model     prompt                                            
llama     acknowledge         38.8          35.7         5.7
          baseline            38.7          35.8         5.4
          ignore              38.4          35.7         5.2
med_gemma acknowledge         68.3          57.2        20.3
          baseline            68.3          57.2        19.8
          ignore              68.1          57.1        19.5
qwen      acknowledge         31.0          24.8         8.9
          baseline            29.9          23.3         9.2
          ignore              30.0          23.5         8.7

=== by model (pooled over prompts) ===
           any_problem  contradicted  fabricated
model                                           
llama             38.6          35.7         5.4
med_gemma         68.2          57.2        19.9
qwen              30.3

In [2]:
import json
import numpy as np
import pandas as pd

df = pd.read_csv("Control_RunLevel_Data_NewJudge.csv", low_memory=False)

# --- parse the judge's problem list ---
def parse(p):
    if isinstance(p, str):
        try:
            return json.loads(p)
        except Exception:
            return None
    return None

df["problems"] = df["fidelity_problems"].apply(parse)
judged = df["problems"].notna()          # runs the judge could evaluate

def has_type(probs, t):
    return float(any(x.get("type") == t for x in probs)) if isinstance(probs, list) else np.nan

df["any_problem"]  = df["problems"].apply(lambda p: float(len(p) > 0) if isinstance(p, list) else np.nan)
df["contradicted"] = df["problems"].apply(lambda p: has_type(p, "CONTRADICTED"))
df["fabricated"]   = df["problems"].apply(lambda p: has_type(p, "FABRICATED"))
df.loc[~judged, ["any_problem", "contradicted", "fabricated"]] = np.nan

cols = ["any_problem", "contradicted", "fabricated"]
print("=== % of explanations with at least one problem, by model & prompt ===")
print((df.groupby(["model", "prompt", "condition"])[cols].mean() * 100).round(1))

print("\n=== by model (pooled over prompts) ===")
print((df.groupby(["model", "condition"])[cols].mean() * 100).round(1))

print("\n=== contradicted-only, by model & vignette (pooled over prompts) ===")
print((df.groupby(["model", "vignette_id", "condition"])["contradicted"].mean() * 100).round(1).unstack(0))

# --- export flagged examples for the two manual checks ---
# check = df[judged & (df["any_problem"] == 1) &
#            (((df.model == "qwen") & (df.vignette_id == 1)) | (df.vignette_id == 5))]
# sample = check.groupby(["model", "vignette_id"]).head(10)
# rows = []
# for r in sample.itertuples():
#     for p in r.problems:
#         rows.append({"model": r.model, "vignette_id": r.vignette_id, "prompt": r.prompt,
#                     "response_n": r.response_n,
#                      "type": p.get("type"), "quote": p.get("quote"),
#                      "case_text_quote": p.get("case_text_quote")})
# pd.DataFrame(rows).to_csv("fidelity_flags_to_check.csv", index=False)
# print(f"\nExported {len(rows)} flagged statements to fidelity_flags_to_check.csv")

=== % of explanations with at least one problem, by model & prompt ===
                                 any_problem  contradicted  fabricated
model     prompt      condition                                       
llama     acknowledge grey_rect         38.3          35.3         4.8
                      no_photo          40.4          37.3         6.8
          baseline    grey_rect         38.0          35.9         3.9
                      no_photo          39.5          36.5         5.9
          ignore      grey_rect         39.4          36.2         5.5
                      no_photo          37.9          35.4         4.8
med_gemma acknowledge grey_rect         69.2          57.0        22.2
                      no_photo          70.4          60.1        20.0
          baseline    grey_rect         68.5          55.9        20.5
                      no_photo          71.7          59.5        22.2
          ignore      grey_rect         70.4          59.6        19.1
      

In [3]:
import pandas as pd

RUN_DATA = "Control_RunLevel_Data_NewJudge.csv"

print("Loading run-level CSV...")
df = pd.read_csv(RUN_DATA)
df

df = pd.read_csv(RUN_DATA)
    # 4. Recalculate the Final EE Tables
target_cols = ['fidelity_passed']

print("--- 1. NEW FILTERED EE (By Model & Prompt) ---")
agg_model_prompt = df.groupby(['model', 'prompt', 'condition'])[target_cols].mean() * 100
print(agg_model_prompt.round(2).to_string())

print("--- 1. NEW FILTERED EE (By Model & Prompt) ---")
# agg_model_prompt = df.groupby(['model', 'prompt', 'vignette_id'])[target_cols].mean() * 100
# print(agg_model_prompt.round(2).to_string())
# agg_model_prompt.to_csv("EE_Model_Prompt_Filtered_new.csv")


import json

def n_contradicted(p):
    try:
        probs = json.loads(p) if isinstance(p, str) else (p or [])
        return sum(1 for x in probs if x.get("type") == "CONTRADICTED")
    except Exception:
        return float("nan")

df["has_contradiction"] = (df["fidelity_problems"].apply(n_contradicted) > 0).astype(float)
df.loc[df["fidelity_passed"].isna(), "has_contradiction"] = float("nan")
print((df.groupby(["model", "prompt",'condition'])["has_contradiction"].mean() * 100).round(1))


Loading run-level CSV...
--- 1. NEW FILTERED EE (By Model & Prompt) ---
                                fidelity_passed
model     prompt      condition                
llama     acknowledge grey_rect       61.724138
                      no_photo        59.620362
          baseline    grey_rect            62.0
                      no_photo        60.542432
          ignore      grey_rect        60.59246
                      no_photo        62.053571
med_gemma acknowledge grey_rect       30.761905
                      no_photo        29.551954
          baseline    grey_rect       31.495327
                      no_photo        28.262911
          ignore      grey_rect       29.626168
                      no_photo         30.68933
qwen      acknowledge grey_rect        70.32967
                      no_photo        68.971332
          baseline    grey_rect       71.717172
                      no_photo        71.571906
          ignore      grey_rect       69.386039
                